In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from diffusers import UNet3DConditionModel, DDPMScheduler

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [2]:
# ------------------------------------------------------------
# 1) Device
# ------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
# ------------------------------------------------------------
# 2) Dummy conditional data (3D)
#    Diffusers UNet3DConditionModel expects:
#      sample shape = (B, C, frames, H, W)
# (frames is effectively your "depth" / #slices / #timesteps)
# ------------------------------------------------------------
B = 4
cond_channels = 1   # e.g., your conditioning volume x
target_channels = 1 # e.g., your target volume y
F, H, W = 16, 64, 64  # frames/depth, height, width

x = torch.randn(B, cond_channels, F, H, W, device=device)      # conditioning
y = torch.randn(B, target_channels, F, H, W, device=device)    # target

In [4]:
# ------------------------------------------------------------
# 3) "Original U-Net style" 3D encoder-decoder (no cross-attn blocks)
#    We still use UNet3DConditionModel, but choose DownBlock3D / UpBlock3D everywhere.
#    Note: UNet3DConditionModel forward requires encoder_hidden_states; we pass a dummy tensor.
# ------------------------------------------------------------
cross_attention_dim = 1  # small because we're not really using cross-attn here

model = UNet3DConditionModel(
    sample_size=None,  # optional; can be left None
    in_channels=target_channels + cond_channels,  # concat(noisy_y, x)
    out_channels=target_channels,                 # predict noise for y
    layers_per_block=2,
    block_out_channels=(64, 128, 256, 512),
    down_block_types=(
        "DownBlock3D",
        "DownBlock3D",
        "DownBlock3D",
        "DownBlock3D",
    ),
    up_block_types=(
        "UpBlock3D",
        "UpBlock3D",
        "UpBlock3D",
        "UpBlock3D",
    ),
    norm_num_groups=8,          # must divide channels nicely; 8 is usually safe
    cross_attention_dim=cross_attention_dim,
    attention_head_dim=8,
).to(device)

In [5]:
# ------------------------------------------------------------
# 4) Noise scheduler + optimizer + loss
#    Your .py uses add_noise() + MSE(noise_pred, noise) :contentReference[oaicite:3]{index=3}
# ------------------------------------------------------------
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)
loss_fn = nn.MSELoss()
opt = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

In [6]:
# ------------------------------------------------------------
# 5) One training step (dummy)
# ------------------------------------------------------------
model.train()

noise = torch.randn_like(y)
timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (B,), device=device).long()

noisy_y = noise_scheduler.add_noise(y, noise, timesteps)

# conditional concat, exactly like your 2D pattern: net_input = cat(noisy_y, x) :contentReference[oaicite:4]{index=4}
net_input = torch.cat([noisy_y, x], dim=1)  # (B, target+cond, F, H, W)

# UNet3DConditionModel forward requires encoder_hidden_states:
# use a dummy "sequence length 1" tensor (B, 1, cross_attention_dim)
encoder_hidden_states = torch.zeros(B, 1, cross_attention_dim, device=device)

noise_pred = model(
    sample=net_input,
    timestep=timesteps,
    encoder_hidden_states=encoder_hidden_states,
).sample

loss = loss_fn(noise_pred, noise)

opt.zero_grad(set_to_none=True)
loss.backward()
opt.step()

print("loss:", float(loss))

loss: 1.1440998315811157


/tmp/ipython-input-3126192367.py:30: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("loss:", float(loss))


In [7]:
# ------------------------------------------------------------
# 6) Very small sampling example (conditional)
# ------------------------------------------------------------
model.eval()
num_inference_steps = 10
noise_scheduler.set_timesteps(num_inference_steps, device=device)

# start from pure noise for y, condition on x
pred = torch.randn_like(y)

for t in noise_scheduler.timesteps:
    net_input = torch.cat([pred, x], dim=1)
    with torch.no_grad():
        residual = model(
            sample=net_input,
            timestep=t,
            encoder_hidden_states=encoder_hidden_states,
        ).sample
    pred = noise_scheduler.step(residual, t, pred).prev_sample

print("sampled pred shape:", pred.shape)

sampled pred shape: torch.Size([4, 1, 16, 64, 64])
